# 《神医平哥》短剧出片机 · ComfyUI + MiniMax H3（BF16 满血版）一键笔记本

**这台笔记本干什么**：在 Google Colab 上一键装好 ComfyUI + MiniMax H3 满血权重，打开一个网页画布，照着《神医平哥》分镜单一镜一镜生成带声音的短剧镜头（图生视频）。

## ⚠️ 运行前必做（顺序不能错）

1. 顶部菜单 **代码执行程序 → 更改运行时类型**：硬件加速器选 **G4 GPU**，**High-RAM 打开**，保存。
2. 然后从上到下 **逐格运行** 本笔记本（点每格左边的 ▶）。
3. **费用提醒**：G4 按连接时长烧计算单元（本账号实测约 10.05 CU/小时，余额 200 CU）。第一次运行要下载约 **124 GB** 权重，是最耗时的环节；权重会存进 Google Drive，以后重开不用再下。**拍完务必在菜单里「断开连接并删除运行时」，不然会一直烧额度。**

## 流程一览

检查显卡 → 挂载云盘 → 装 ComfyUI 并把模型/工作流/成片目录软链到云盘 → 下载 H3 满血权重到云盘（已有自动跳过）→ 启动 ComfyUI 并生成公网链接 → 点链接进画布出片。


In [ ]:
#@title 1️⃣ 检查显卡（必须是 G4 / RTX PRO 6000，显存约 96GB）
!nvidia-smi
import torch
print("PyTorch:", torch.__version__, "| CUDA 可用:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("显卡:", torch.cuda.get_device_name(0), "| 显存(GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))


In [ ]:
#@title 2️⃣ 挂载 Google Drive（权重、工作流、成片都存这里，关机不丢）
import os
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/Pingge_ComfyUI"   # ComfyUI 持久化根目录
for sub in ["models/diffusion_models", "models/text_encoders", "models/vae", "models/loras",
            "models/embeddings", "models/model_patches",
            "user/default/workflows", "output"]:
    os.makedirs(os.path.join(DRIVE_ROOT, sub), exist_ok=True)

KF_DIR = "/content/drive/MyDrive/Pingge_Shenyi/keyframes"  # 分镜关键帧图存放处（镜7关键帧放这里）
os.makedirs(KF_DIR, exist_ok=True)
print("✅ 云盘已挂载")
print("权重/工作流/成片目录:", DRIVE_ROOT)
print("关键帧目录:", KF_DIR)


In [ ]:
#@title 3️⃣ 安装 ComfyUI + 核心插件，并把模型/工作流/输出目录软链到云盘（已装自动跳过）
import os, subprocess, sys, shutil

COMFY = "/content/ComfyUI"
DRIVE_ROOT = "/content/drive/MyDrive/Pingge_ComfyUI"

def link_dir(drive_dir, local_dir):
    """把 ComfyUI 的本地目录替换为指向云盘的软链（做法与 EasyColab 相同：程序在本地盘、数据在云盘）。"""
    os.makedirs(drive_dir, exist_ok=True)
    if os.path.islink(local_dir):
        return
    if os.path.isdir(local_dir):
        # 本地已有文件先搬去云盘再换软链，避免丢东西
        for name in os.listdir(local_dir):
            src = os.path.join(local_dir, name)
            dst = os.path.join(drive_dir, name)
            if not os.path.exists(dst):
                shutil.move(src, dst)
        shutil.rmtree(local_dir)
    elif os.path.exists(local_dir):
        os.remove(local_dir)
    os.makedirs(os.path.dirname(local_dir), exist_ok=True)
    os.symlink(drive_dir, local_dir)

# --- 克隆 ComfyUI 主程序（本地高速盘）---
if not os.path.exists(os.path.join(COMFY, "main.py")):
    print("[1/4] 正在克隆 ComfyUI 官方仓库...")
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI.git", COMFY], check=True)
    # 安装依赖：剔除 torch 相关行，保留 Colab 预装的 G4(Blackwell) 定制版 PyTorch 不被覆盖（EasyColab 同款保护做法）
    req_lines = open(os.path.join(COMFY, "requirements.txt"), encoding="utf-8").read().splitlines()
    protected = ("torch", "torchvision", "torchaudio")
    safe = [ln for ln in req_lines if ln.strip() and not ln.strip().startswith("#")
            and ln.strip().split("=")[0].split(">")[0].split("<")[0].strip().lower() not in protected]
    with open("/tmp/comfy_reqs_safe.txt", "w") as f:
        f.write("\n".join(safe) + "\n")
    print("[2/4] 正在安装 ComfyUI 依赖（约几分钟）...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/tmp/comfy_reqs_safe.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "gguf", "opencv-python-headless", "imageio-ffmpeg", "gitpython", "toml", "huggingface_hub"], check=False)
else:
    print("ComfyUI 已存在，跳过克隆与依赖安装")

# --- 预装两个核心插件（H3 本身是 ComfyUI 原生节点、不需要插件；这两个是管理器和视频辅助）---
default_nodes = [
    ("ComfyUI-Manager", "https://github.com/ltdrdata/ComfyUI-Manager.git"),
    ("ComfyUI-VideoHelperSuite", "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git"),
]
for name, url in default_nodes:
    dst = os.path.join(COMFY, "custom_nodes", name)
    if not os.path.exists(dst):
        print(f"[3/4] 正在安装插件 {name} ...")
        subprocess.run(["git", "clone", "--depth", "1", url, dst], check=False)

# --- 软链：models 子目录 / user（工作流） / output（成片）全部指向云盘 ---
print("[4/4] 正在建立云盘软链...")
for sub in ["diffusion_models", "text_encoders", "vae", "loras", "embeddings", "model_patches"]:
    link_dir(os.path.join(DRIVE_ROOT, "models", sub), os.path.join(COMFY, "models", sub))
link_dir(os.path.join(DRIVE_ROOT, "user"), os.path.join(COMFY, "user"))
link_dir(os.path.join(DRIVE_ROOT, "output"), os.path.join(COMFY, "output"))
print("✅ ComfyUI 安装完成，模型/工作流/成片都会存进云盘")


In [ ]:
#@title 4️⃣ 下载 MiniMax H3 满血(BF16)权重到云盘（共约 124GB，首次最久；已有文件自动跳过、可断点续传）
import os
from huggingface_hub import hf_hub_download

REPO = "Comfy-Org/MiniMax-H3"   # ComfyUI 官方重打包仓库
DRIVE_ROOT = "/content/drive/MyDrive/Pingge_ComfyUI"
MODELS_DIR = os.path.join(DRIVE_ROOT, "models")

# (仓库内路径, 精确字节数) —— 字节数来自 HuggingFace 官方 API 核对
FILES = [
    ("diffusion_models/minimax_h3_fl2va_bf16.safetensors",          66280487368),  # 主干 DiT 66.28GB
    ("text_encoders/qwen3vl_32b_minimax_h3_bf16.safetensors",       51506295256),  # 文本编码器 51.51GB
    ("vae/minimax_h3_video_vae_fp16.safetensors",                    5207808496),  # 视频 VAE 5.21GB
    ("vae/minimax_h3_audio_vae_fp32.safetensors",                      605254808),  # 音频 VAE 0.61GB
    ("loras/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors", 1956193000),  # 可选：8步快速试错 LoRA 1.96GB
]
total_gb = sum(s for _, s in FILES) / 1e9
print(f"本次需备齐权重共约 {total_gb:.1f} GB（已存在的会自动跳过）")

for path, size in FILES:
    dest = os.path.join(MODELS_DIR, path)
    if os.path.exists(dest) and os.path.getsize(dest) == size:
        print(f"⏭️ 已存在，跳过：{path}（{size/1e9:.2f} GB）")
        continue
    print(f"⬇️ 正在下载：{path}（{size/1e9:.2f} GB）……这一步最久，请保持页面开着")
    hf_hub_download(repo_id=REPO, filename=path, local_dir=MODELS_DIR)
    print(f"✅ 完成：{path}")

# 顺带把官方图生视频工作流模板存进云盘（进 ComfyUI 后可在左侧工作流列表直接打开）
import urllib.request
tpl_url = "https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/templates/video_minimax_h3_i2v.json"
tpl_dst = os.path.join(DRIVE_ROOT, "user/default/workflows/video_minimax_h3_i2v.json")
if not os.path.exists(tpl_dst):
    urllib.request.urlretrieve(tpl_url, tpl_dst)
    print("✅ 官方 I2V 工作流模板已保存：video_minimax_h3_i2v.json")
else:
    print("⏭️ 工作流模板已存在，跳过")

# 若关键帧已放在云盘关键帧目录，自动拷进 ComfyUI 的 input 供 LoadImage 节点选用
KF_DIR = "/content/drive/MyDrive/Pingge_Shenyi/keyframes"
import shutil
os.makedirs(os.path.join("/content/ComfyUI", "input"), exist_ok=True)
for name in os.listdir(KF_DIR):
    if name.lower().endswith((".png", ".jpg", ".jpeg", ".webp")):
        shutil.copy(os.path.join(KF_DIR, name), os.path.join("/content/ComfyUI", "input", name))
        print(f"🖼️ 关键帧已就位：{name}")
print("🎉 权重与工作流全部就绪，下一步启动 ComfyUI")


In [ ]:
#@title 5️⃣ 启动 ComfyUI 并生成公网链接（跑完后点输出里的链接进画布）
import os, re, subprocess, sys, threading, time, urllib.request

COMFY = "/content/ComfyUI"
PORT = 8188

# 杀掉可能残留的旧进程（重跑本格时用）
subprocess.run(["pkill", "-f", "ComfyUI/main.py"], check=False)
subprocess.run(["pkill", "-f", "cloudflared.*8188"], check=False)
time.sleep(1)

# --- 启动 ComfyUI（参数与 EasyColab 源码一致）---
comfy = subprocess.Popen(
    [sys.executable, "-u", "main.py", "--listen", "127.0.0.1", "--port", str(PORT),
     "--enable-cors-header", "*", "--preview-method", "auto"],
    cwd=COMFY, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

def forward(proc, tag):
    for line in proc.stdout:
        print(f"[{tag}] {line}", end="")

threading.Thread(target=forward, args=(comfy, "ComfyUI"), daemon=True).start()

opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))
ready = False
for _ in range(120):
    if comfy.poll() is not None:
        raise RuntimeError(f"ComfyUI 启动失败（退出码 {comfy.returncode}），看上方日志")
    try:
        with opener.open(f"http://127.0.0.1:{PORT}/system_stats", timeout=1.5) as r:
            if r.status == 200:
                ready = True
                break
    except Exception:
        pass
    time.sleep(1)
if not ready:
    raise RuntimeError("等待 ComfyUI 就绪超时，看上方日志")
print("✅ ComfyUI 已在本机 8188 端口就绪，正在开公网隧道……")

# --- 下载 cloudflared（官方发布）并开快速隧道 ---
CF = "/content/cloudflared"
if not os.path.exists(CF):
    subprocess.run(["curl", "-sL", "-o", CF,
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    subprocess.run(["chmod", "+x", CF], check=True)

tunnel = subprocess.Popen([CF, "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
url_box = [""]
cf_re = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")

def watch(proc):
    for line in proc.stdout:
        print(f"[隧道] {line}", end="")
        m = cf_re.search(line)
        if m and not url_box[0]:
            url_box[0] = m.group(0)

threading.Thread(target=watch, args=(tunnel,), daemon=True).start()
for _ in range(90):
    if url_box[0]:
        break
    time.sleep(0.5)

if url_box[0]:
    print("\n" + "=" * 60)
    print("🎉 点这个链接进 ComfyUI 画布（每次重启链接都会变，别外传，别人能蹭你的显卡）：")
    print(url_box[0])
    print("=" * 60)
else:
    print("⚠️ 隧道链接还没刷出来，等 1 分钟后看上方[隧道]日志里的 https://xxxx.trycloudflare.com 链接")


# 🎬 进画布后怎么出片（以第一集镜 7「血滴针盒」为例）

1. **打开工作流**：点上一格给出的链接进 ComfyUI → 左侧 **Workflows** 列表里选 `video_minimax_h3_i2v` 打开（这是官方图生视频模板，已存进你的云盘）。也可以走菜单 Template Library → Video → MiniMax H3 Image to Video。
2. **必改 4 个模型下拉**（模板默认选的是量化版，不改就不是满血）：在画布中间那个大节点 `MiniMaxH3ImageToVideo` 上依次改——
   - `unet_name` → **minimax_h3_fl2va_bf16.safetensors**
   - `clip_name` → **qwen3vl_32b_minimax_h3_bf16.safetensors**
   - `vae_name` → **minimax_h3_video_vae_fp16.safetensors**
   - `vae_name_1`（音频 VAE）→ **minimax_h3_audio_vae_fp32.safetensors**
   - `turbo_mode` 保持 **false**（满血 20 步）；只想快速看个大概时可打开 turbo（8 步，出片快、质量略降）。
3. **放首帧图**：在 `LoadImage` 节点选镜 7 关键帧。建议先把关键帧存到云盘 **`MyDrive/Pingge_Shenyi/keyframes/EP01_shot7_keyframe.png`**，下载格会自动把它拷进画布的图库；也可以在节点上直接点 upload 上传。
   - ⚠️ 关键帧本身必须是 **9:16 竖屏**：H3 会把首帧直接拉伸到画布尺寸，比例不对画面会变形。
4. **设画面尺寸**：`ResolutionSelector` 节点比例选 **9:16**，Megapixels 设 **1.0**（约 768×1344，是 H3 原生上限）。
5. **设时长**：大节点上的 `duration` 填 **8**（秒；模型会自动对齐到 24fps 的帧数网格）。
6. **粘贴提示词**：把下面这段原文粘进大节点的 prompt 文本框——

```text
Extreme close-up, slow push-in. A drop of blood rests on the inner lid of an open dark ebony acupuncture box on a hospital corridor floor, nine silver needles in the tray. Golden glowing vein patterns spread from the blood drop across the black wood like a growing tree of light; the silver needles tremble slightly and glow with soft golden light; tiny golden particles rise into the air. Cold blue hospital lighting, cinematic, shallow depth of field, semi-realistic Chinese anime style, no people. Vertical 9:16, 8 seconds.
```

7. **点 Run 生成**：第一次生成要先把 124GB 权重从云盘读进显存/内存，会明显慢；之后同参数重抽会快很多。出片在右侧 SaveVideo 预览，文件自动存进云盘 `MyDrive/Pingge_ComfyUI/output/video/`。
8. **收尾**：全部拍完，回 Colab 菜单 **代码执行程序 → 断开连接并删除运行时**，停止烧 CU。下次再开，重复运行本笔记本即可（安装和下载都会自动跳过已完成的部分）。

> 镜 7 这一镜就是拿来校准的：记下它实际生成耗时，后面整集的 CU 预算就能算准了。
